# Import Modules

In [1]:
import importlib
import os
import sys

import joblib
import numpy as np
import pandas as pd
import polars as pl
from sklearn.model_selection import train_test_split

In [2]:
os.chdir("../")
sys.path.insert(0, os.getcwd())

In [3]:
from morai.experience import charters, experience
from morai.forecast import metrics, preprocessors
from morai.models import core
from morai.utils import custom_logger, helpers

In [4]:
# from morai.models import r

In [5]:
logger = custom_logger.setup_logging(__name__)

In [6]:
# update log level if wanting more logging
custom_logger.set_log_level("INFO")

In [7]:
pd.options.display.float_format = "{:,.2f}".format

In [8]:
# default is "plotly_mimetype+notebook", however that takes up space.
# "plotly_mimetype+notebook_connected" seems to save space
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

# Data

In [9]:
pl_parquet_path = helpers.FILES_PATH / "dataset" / "raw_mortality_grouped_1223.parquet"

In [10]:
# reading in the dataset
pl.enable_string_cache()
lzdf = pl.scan_parquet(
    pl_parquet_path,
)

/tmp/ipykernel_3251/316789095.py:2: DeprecationWarning: the string cache has been replaced by pl.Categories
  pl.enable_string_cache()


In [11]:
initial_row_count = lzdf.select(pl.len()).collect().item()
print(
    f"row count: {initial_row_count:,} \n"
    f"exposures: {lzdf.select([pl.col('amount_exposed').sum()]).collect()[0,0]:,}"
)

row count: 39,595,138 
exposures: 170,725,471,878,137.56


In [12]:
grouped_df = lzdf.collect()

In [13]:
grouped_df = grouped_df.to_pandas()

# Preparing Data

## Filter

In [14]:
model_data = grouped_df[
    (grouped_df["issue_age"] >= 18)
    & (grouped_df["issue_age"] <= 95)
    & (grouped_df["observation_year"] >= 2012)
    & ~(grouped_df["soa_post_lvl_ind"].isin(["PLT"]))
    & (grouped_df["insurance_plan"]!="Other")
    & (grouped_df["smoker_status"]!="U")
].copy()
model_data = model_data.reset_index(drop=True)

In [15]:
model_data = model_data[
    (model_data["attained_age"] >= 50)
    & (model_data["attained_age"] <= 95)
    & (model_data["issue_age"] >= 30)
    & (model_data["issue_age"] <= 80)
].copy()

In [16]:
# model_data = model_data[(model_data["observation_year"] <= 2019)].copy()
# model_data = model_data[~model_data["observation_year"].isin([2020, 2021, 2022])].copy()

In [17]:
del grouped_df

## Calculated Fields

In [18]:
model_data["capped_duration"] = model_data["duration"].clip(upper=26)
model_data["qx_log_raw"] = np.log(model_data["qx_raw"] + 1)
binned_face_dict = {
    "01: 0 - 9,999": "01: 0 - 24,999",
    "02: 10,000 - 24,999": "01: 0 - 24,999",
    "03: 25,000 - 49,999": "02: 25,000 - 99,999",
    "04: 50,000 - 99,999": "02: 25,000 - 99,999",
    "05: 100,000 - 249,999": "03: 100,000 - 249,999",
    "06: 250,000 - 499,999": "04: 250,000 - 4,999,999",
    "07: 500,000 - 999,999": "04: 250,000 - 4,999,999",
    "08: 1,000,000 - 2,499,999": "04: 250,000 - 4,999,999",
    "09: 2,500,000 - 4,999,999": "04: 250,000 - 4,999,999",
    "10: 5,000,000 - 9,999,999": "05: 5,000,000+",
    "11: 10,000,000+": "05: 5,000,000+",
}
model_data["binned_face"] = model_data["face_amount_band"].map(binned_face_dict)
model_data["binned_face"] = model_data["binned_face"].astype("category")

## Feature Dictionary

In [19]:
feature_dict = {
    "target": ["qx_raw"],
    "weight": ["amount_exposed"],
    "passthrough": ["attained_age", "duration", "observation_year"],
    "ordinal": [
        "sex",
        "smoker_status",
    ],
    "ohe": [
        "binned_face",
        "insurance_plan",
        "class_enh",
    ],
    "nominal": [],
}

## Model Results Dictionary

In [20]:
metric_cols = ["ae", "smape", "r2_score", "root_mean_squared_error", "aic", "shape"]
model_results = metrics.ModelResults(metrics=metric_cols)

### VBT15

In [21]:
model_name = "vbt15"

In [22]:
scorecard = model_results.get_scorecard(
    y_true_train=model_data["death_claim_amount"],
    y_pred_train=model_data[f"exp_amt_{model_name}"],
    weights_train=None,
)
model_results.add_model(
    model_name=model_name,
    data_path=pl_parquet_path,
    data_shape=model_data.shape,
    preprocess_dict=None,
    model_params=None,
    scorecard=scorecard,
    importance=None,
)

 2026-07-12 13:57:26 | morai.forecast.metrics | INFO     | Adding model 'vbt15' 


# Forecasting Models

In [23]:
# load
model_build = False
model_load = True
model_save = False

## GLM

In [24]:
model_name = "glm_1219_amt"

preprocess_dict = preprocessors.preprocess_data(
    model_data,
    feature_dict=feature_dict,
    standardize=False,
    add_constant=True,
)

X = preprocess_dict["X"]
y = preprocess_dict["y"]
weights = preprocess_dict["weights"]
mapping = preprocess_dict["mapping"]
md_encoded = preprocess_dict["md_encoded"]
model_features = preprocess_dict["model_features"]

 2026-07-12 13:57:26 | morai.forecast.preprocessors | INFO     | model target: ['qx_raw'] 
 2026-07-12 13:57:26 | morai.forecast.preprocessors | INFO     | model weights: ['amount_exposed'] 
 2026-07-12 13:57:26 | morai.forecast.preprocessors | INFO     | adding a constant column to the data 
 2026-07-12 13:57:27 | morai.forecast.preprocessors | INFO     | passthrough - (generally numeric): ['duration', 'observation_year', 'attained_age', 'constant'] 
 2026-07-12 13:57:27 | morai.forecast.preprocessors | INFO     | ordinal - ordinal encoded: ['sex', 'smoker_status'] 
 2026-07-12 13:57:32 | morai.forecast.preprocessors | INFO     | ohe - one hot encoded (dropping first col): ['class_enh', 'binned_face', 'insurance_plan'] 


In [25]:
GLM = core.GLM()
GLM.model = joblib.load(helpers.FILES_PATH / "models" / f"{model_name}.joblib")
logger.info(f"loaded model '{model_name}'. type: {type(GLM.model)}")
GLM.is_fitted_ = True
model_params = {"weights": True, "r_style": False}
model_params.update({"family": GLM.model.family})

predictions = GLM.predict(X)
print(f"NA values: {np.isnan(predictions).sum()}")

 2026-07-12 13:58:00 | __main__ | INFO     | loaded model 'glm_1219_amt'. type: <class 'statsmodels.genmod.generalized_linear_model.GLMResultsWrapper'> 
NA values: 0


In [26]:
model_data = experience.calc_qx_exp_ae(
    model_data=model_data,
    predictions=predictions,
    model_name=model_name,
    exposure_col="amount_exposed",
    actual_col="death_claim_amount",
)

odds = GLM.get_odds(display=False)
importance = core.ModelWrapper(GLM.model).get_importance()

scorecard = model_results.get_scorecard(
    y_true_train=y,
    y_pred_train=GLM.model.predict(X),
    weights_train=weights,
    y_true_test=y,
    y_pred_test=GLM.model.predict(X),
    weights_test=weights,
    model=GLM.model,
)
model_results.add_model(
    model_name=model_name,
    data_path=pl_parquet_path,
    data_shape=model_data.shape,
    preprocess_dict=preprocess_dict,
    model_params=model_params,
    scorecard=scorecard,
    importance=importance,
)

 2026-07-12 13:59:17 | morai.models.core | INFO     | generating odds ratio from model 
 2026-07-12 14:02:07 | morai.forecast.metrics | INFO     | Adding model 'glm_1219_amt' 


## CatBoost - 1219

In [27]:
model_name = "cat_1219_amt"

preprocess_dict = preprocessors.preprocess_data(
    model_data,
    feature_dict=feature_dict,
    standardize=False,
    preset="pass",
)

X = preprocess_dict["X"]
y = preprocess_dict["y"]
weights = preprocess_dict["weights"]
mapping = preprocess_dict["mapping"]
md_encoded = preprocess_dict["md_encoded"]
model_features = preprocess_dict["model_features"]

cat_features = feature_dict["ordinal"] + feature_dict["ohe"] + feature_dict["nominal"]
cat_features = list(set(cat_features) & set(model_features))

 2026-07-12 14:02:07 | morai.forecast.preprocessors | INFO     | using 'pass' preset which makes all features passthrough 
 2026-07-12 14:02:07 | morai.forecast.preprocessors | INFO     | model target: ['qx_raw'] 
 2026-07-12 14:02:07 | morai.forecast.preprocessors | INFO     | model weights: ['amount_exposed'] 
 2026-07-12 14:02:08 | morai.forecast.preprocessors | INFO     | passthrough - (generally numeric): ['duration', 'observation_year', 'attained_age', 'sex', 'smoker_status', 'class_enh', 'binned_face', 'insurance_plan'] 


In [28]:
from catboost import CatBoostRegressor

model = joblib.load(helpers.FILES_PATH / "models" / f"{model_name}.joblib")
logger.info(f"loaded model '{model_name}'. type: {type(model)}")
model_params = {"weights": True}
model_params.update(model.get_params())

predictions = model.predict(X)

 2026-07-12 14:04:09 | __main__ | INFO     | loaded model 'cat_1219_amt'. type: <class 'catboost.core.CatBoostRegressor'> 


In [29]:
model_data = experience.calc_qx_exp_ae(
    model_data=model_data,
    predictions=predictions,
    model_name=model_name,
    exposure_col="amount_exposed",
    actual_col="death_claim_amount",
)

importance = core.ModelWrapper(model).get_importance()

scorecard = model_results.get_scorecard(
    y_true_train=y,
    y_pred_train=model.predict(X),
    weights_train=weights,
    y_true_test=y,
    y_pred_test=model.predict(X),
    weights_test=weights,
    model=None,
)
model_results.add_model(
    model_name=model_name,
    data_path=pl_parquet_path,
    data_shape=model_data.shape,
    preprocess_dict=preprocess_dict,
    model_params=model_params,
    scorecard=scorecard,
    importance=importance,
)

 2026-07-12 14:04:23 | morai.forecast.metrics | INFO     | Adding model 'cat_1219_amt' 


## CatBoost - 121923

In [30]:
model_name = "cat_121923_amt"

preprocess_dict = preprocessors.preprocess_data(
    model_data,
    feature_dict=feature_dict,
    standardize=False,
    preset="pass",
)

X = preprocess_dict["X"]
y = preprocess_dict["y"]
weights = preprocess_dict["weights"]
mapping = preprocess_dict["mapping"]
md_encoded = preprocess_dict["md_encoded"]
model_features = preprocess_dict["model_features"]

cat_features = feature_dict["ordinal"] + feature_dict["ohe"] + feature_dict["nominal"]
cat_features = list(set(cat_features) & set(model_features))

 2026-07-12 14:04:23 | morai.forecast.preprocessors | INFO     | using 'pass' preset which makes all features passthrough 
 2026-07-12 14:04:23 | morai.forecast.preprocessors | INFO     | model target: ['qx_raw'] 
 2026-07-12 14:04:23 | morai.forecast.preprocessors | INFO     | model weights: ['amount_exposed'] 
 2026-07-12 14:04:24 | morai.forecast.preprocessors | INFO     | passthrough - (generally numeric): ['duration', 'observation_year', 'attained_age', 'sex', 'smoker_status', 'class_enh', 'binned_face', 'insurance_plan'] 


In [31]:
from catboost import CatBoostRegressor

model = joblib.load(helpers.FILES_PATH / "models" / f"{model_name}.joblib")
logger.info(f"loaded model '{model_name}'. type: {type(model)}")
model_params = {"weights": True}
model_params.update(model.get_params())

predictions = model.predict(X)

 2026-07-12 14:06:01 | __main__ | INFO     | loaded model 'cat_121923_amt'. type: <class 'catboost.core.CatBoostRegressor'> 


In [32]:
model_data = experience.calc_qx_exp_ae(
    model_data=model_data,
    predictions=predictions,
    model_name=model_name,
    exposure_col="amount_exposed",
    actual_col="death_claim_amount",
)

importance = core.ModelWrapper(model).get_importance()

scorecard = model_results.get_scorecard(
    y_true_train=y,
    y_pred_train=model.predict(X),
    weights_train=weights,
    y_true_test=y,
    y_pred_test=model.predict(X),
    weights_test=weights,
    model=None,
)
model_results.add_model(
    model_name=model_name,
    data_path=pl_parquet_path,
    data_shape=model_data.shape,
    preprocess_dict=preprocess_dict,
    model_params=model_params,
    scorecard=scorecard,
    importance=importance,
)

 2026-07-12 14:06:15 | morai.forecast.metrics | INFO     | Adding model 'cat_121923_amt' 


## Neural

In [33]:
feature_dict = {
    "target": ["qx_raw"],
    "weight": ["amount_exposed"],
    "passthrough": [
        "face_amount_band",
    ],
    "ordinal": [
        "sex",
        "smoker_status",
        "observation_year",
    ],
    "nominal": [],
    "ohe": [
        "class_enh",
        "insurance_plan",
    ],
    "spline": {
        "attained_age": {"n_knots": 8, "degree": 3, "knots": "quantile"},
        "duration": {"n_knots": 5, "degree": 3, "knots": "quantile"},
    },
}

In [34]:
model_name = "neural"

preprocess_dict = preprocessors.preprocess_data(
    model_data,
    feature_dict=feature_dict,
    standardize=True,
)

X = preprocess_dict["X"]
y = preprocess_dict["y"]
weights = preprocess_dict["weights"]
mapping = preprocess_dict["mapping"]
md_encoded = preprocess_dict["md_encoded"]
model_features = preprocess_dict["model_features"]
spline_dict = preprocess_dict["spline_dict"]

 2026-07-12 14:06:15 | morai.forecast.preprocessors | INFO     | model target: ['qx_raw'] 
 2026-07-12 14:06:15 | morai.forecast.preprocessors | INFO     | model weights: ['amount_exposed'] 
 2026-07-12 14:06:15 | morai.forecast.preprocessors | INFO     | passthrough - (generally numeric): ['face_amount_band'] 
 2026-07-12 14:06:15 | morai.forecast.preprocessors | INFO     | ordinal - ordinal encoded: ['sex', 'smoker_status', 'observation_year'] 
 2026-07-12 14:06:22 | morai.forecast.preprocessors | INFO     | ohe - one hot encoded (dropping first col): ['class_enh', 'insurance_plan'] 
 2026-07-12 14:06:28 | morai.forecast.preprocessors | INFO     | spline - b-spline basis expansion: ['duration', 'attained_age'] 
 2026-07-12 14:07:15 | morai.forecast.preprocessors | INFO     | standardizing the features with StandardScaler (that excludes OHE) 


In [35]:
model = joblib.load(f"files/models/{model_name}.joblib")
logger.info(f"loaded model '{model_name}'. type: {type(model)}")

predictions = model.predict(X)

 2026-07-12 14:17:59 | __main__ | INFO     | loaded model 'neural'. type: <class 'morai.models.neural.Neural'> 


In [36]:
model_data = experience.calc_qx_exp_ae(
    model_data=model_data,
    predictions=predictions,
    model_name=model_name,
    exposure_col="amount_exposed",
    actual_col="death_claim_amount",
)

scorecard = model_results.get_scorecard(
    y_true_train=y,
    y_pred_train=model.predict(X),
    weights_train=weights,
    y_true_test=y,
    y_pred_test=model.predict(X),
    weights_test=weights,
    model=None,
)
model_results.add_model(
    model_name=model_name,
    data_path=pl_parquet_path,
    data_shape=model_data.shape,
    preprocess_dict=preprocess_dict,
    model_params=model_params,
    scorecard=scorecard,
    importance=None,
)

 2026-07-12 14:23:42 | morai.forecast.metrics | INFO     | Adding model 'neural' 


# Model Results

In [37]:
original_float_format = pd.options.display.float_format
pd.options.display.float_format = "{:,.6f}".format  # Increase to 10 decimal places
model_results.scorecard.sort_values(by=("test", "r2_score"), ascending=False)

model_name    train                                            \
                        ae    smape r2_score root_mean_squared_error   
3  cat_121923_amt 1.035570 1.923462 0.214233          211,982.070994   
2    cat_1219_amt 1.027072 1.923491 0.193543          214,754.831080   
1    glm_1219_amt 1.033426 1.924954 0.123497          223,887.012905   
4          neural 1.051418 1.923588 0.109903          225,616.621741   
0           vbt15 0.911270 1.926736 0.127797          223,337.251619   

                                         test                    \
      shape                      aic       ae    smape r2_score   
3  17897990                      NaN 1.035570 1.923462 0.214233   
2  17897990                      NaN 1.027072 1.923491 0.193543   
1  17897990 1,885,817,368,356.485107 1.033426 1.924954 0.123497   
4  17897990                      NaN 1.051418 1.923588 0.109903   
0  17897990                      NaN      NaN      NaN      NaN   

                                                                      
  root_mean_squared_error                      aic             shape  
3          211,982.070994                      NaN 17,897,990.000000  
2          214,754.831080                      NaN 17,897,990.000000  
1          223,887.012905 1,885,817,368,356.485107 17,897,990.000000  
4          225,616.621741                      NaN 17,897,990.000000  
0                     NaN                      NaN               NaN

In [38]:
pd.options.display.float_format = original_float_format

In [39]:
# write out model results
model_results.save_model()

# write out model data
model_data.to_parquet(helpers.FILES_PATH / "dataset" / "model_data_1223.parquet")

 2026-07-12 14:23:42 | morai.forecast.metrics | INFO     | saving results to model_results.json 


In [40]:
model_data.columns

Index(['observation_year', 'sex', 'smoker_status', 'insurance_plan',
       'issue_age', 'duration', 'face_amount_band', 'issue_year',
       'attained_age', 'soa_post_lvl_ind', 'number_of_pfd_classes',
       'preferred_class', 'amount_exposed', 'policies_exposed',
       'death_claim_amount', 'death_count', 'cen2momp1wmi_byamt',
       'cen2momp2wmi_byamt', 'class_enh', 'qx_vbt15', 'qx_raw', 'qx_log_raw',
       'exp_amt_vbt15', 'ae_vbt15', 'capped_duration', 'binned_face',
       'constant', 'qx_glm_1219_amt', 'exp_glm_1219_amt', 'ae_glm_1219_amt',
       'qx_cat_1219_amt', 'exp_cat_1219_amt', 'ae_cat_1219_amt',
       'qx_cat_121923_amt', 'exp_cat_121923_amt', 'ae_cat_121923_amt',
       'qx_neural', 'exp_neural', 'ae_neural'],
      dtype='object')

# Reload

In [54]:
importlib.reload(charters)

<module 'morai.experience.charters' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\experience\\charters.py'>

In [ ]:
importlib.reload(custom_logger)

In [117]:
importlib.reload(core)

<module 'morai.models.core' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\models\\core.py'>

In [162]:
importlib.reload(helpers)

<module 'morai.utils.helpers' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\utils\\helpers.py'>

In [163]:
importlib.reload(metrics)

<module 'morai.forecast.metrics' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\forecast\\metrics.py'>

In [164]:
importlib.reload(preprocessors)

<module 'morai.forecast.preprocessors' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\forecast\\preprocessors.py'>